# Paraphrase attack (role D)
No logic here: every cell just calls `src/paraphrase.py`. Settings: GPU T4, Internet on, dataset `team20-m4-processed` attached.
Outputs: `outputs/paraphrase/paraphrased.parquet` (give to Nirmal for a new dataset version), `splits/paraphrase.json`, `splits/paraphrase_human.json` and `results/paraphrase_quality.json` (commit these three).

**What this step does:** downloads the project code from GitHub, installs the Python packages, links the attached dataset into `data/processed`, and checks that every id in the split files exists in the data. It is needed because a fresh Kaggle session starts empty.

**What you should see:** the `ls` listing shows `m4_en.parquet`, and the split check finishes without an error.

**How long:** quick, but it depends on the network (clone and pip install need Internet). No measured time is recorded for this step.

In [ ]:
!git clone -b main https://github.com/NIRMAL20062/generated_text_detection.git
%cd generated_text_detection
# If pip breaks torch, comment this line out: Kaggle's preinstalled packages are enough.
!pip install -q -r requirements.txt
# The dataset mounts under /kaggle/input/... (path differs by account), so find it.
!mkdir -p data && ln -sfn "$(dirname "$(find /kaggle/input -name m4_en.parquet | head -1)")" data/processed
!ls -l data/processed/   # must show m4_en.parquet
!python -m src.data --check-splits

## Paraphrase the 500 machine texts (+ 200 human texts for the optional control)

**What this step does:** rewrites the 500 machine texts (and 200 human texts for the optional control) with the T5 paraphraser `humarin/chatgpt_paraphraser_on_T5_base`. This makes the paraphrase attack data: the idea is to see whether detectors still work on rewritten text. It writes `outputs/paraphrase/paraphrased.parquet`, the split files and `results/paraphrase_quality.json`.

**What you should see:** generation progress, then the quality file with BLEU scores and the number of texts (500 machine, 200 human).

**How long:** about 7 minutes on a Kaggle T4 GPU (399.9 s, from `results/paraphrase_quality.json`).

In [ ]:
!python -m src.paraphrase

## Check the splits against the data, then bundle the files to download

**What this step does:** re-checks that all ids in the new split files exist in the data, lists the output files, prints the quality file, and zips the three files to commit plus the parquet into `/kaggle/working/paraphrase_results.zip`.

**What you should see:** the file listing, the quality JSON, and then the zip appearing in the Output tab.

**How long:** quick (no measured time).

In [ ]:
!python -m src.data --check-splits
!ls -lh outputs/paraphrase/paraphrased.parquet splits/paraphrase.json splits/paraphrase_human.json results/paraphrase_quality.json
!cat results/paraphrase_quality.json
!zip -rq /kaggle/working/paraphrase_results.zip outputs/paraphrase/paraphrased.parquet splits/paraphrase.json splits/paraphrase_human.json results/paraphrase_quality.json